In [0]:
df = spark.read.csv("/Volumes/workspace/default/my_volume/train.csv", header=True, inferSchema=True)
df.display()

In [0]:
from pyspark.sql.functions import col

# Remove null values
clean_df = df.dropna()

# Remove invalid data (example: Quantity > 0)
if "Quantity" in df.columns:
    clean_df = clean_df.filter(col("Quantity") > 0)

clean_df.display()

In [0]:
from pyspark.sql.functions import col

clean_df = df.dropna().dropDuplicates()


clean_df.display()

In [0]:
print("Original count:", df.count())
print("After removing duplicates:", clean_df.count())

In [0]:
print(clean_df.columns)

In [0]:
from pyspark.sql.functions import col

clean_df = clean_df.withColumn(
    "Sales",
    col("Sales").cast("double")
)

In [0]:
clean_df = clean_df.filter(col("Sales").isNotNull())

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .load("/Volumes/workspace/default/my_volume/train.csv")

df.display()

In [0]:
print(df.columns)

In [0]:
clean_df = df.dropna().dropDuplicates()

In [0]:
# aggregation
from pyspark.sql.functions import sum

region_sales = clean_df.groupBy("Region") \
    .agg(sum("Sales").alias("Total_Sales"))

region_sales.display()

In [0]:
#gold table
region_sales.write.mode("overwrite").saveAsTable("gold_table")

In [0]:
%sql
select Region, sum(Total_Sales) AS Sales
from gold_table
group by Region
order by Sales DESC;